<div align="center">

# Week 12: 3D User Interfaces, Virtual and Augmented Reality

**Human-Computer Interaction (BIL-429), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-12) · [Lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-12/lecture.html) · [Interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-12/lab.html) · [PDF notes](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-12/Week12_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-12/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also stores the twelve signs of the atrium with their distances, widths and directions, and the scatter of the ray. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import random
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok


# The twelve square signs of the atrium in the order of Part A: name,
# distance d (m), width w (m), azimuth and elevation of the centre (degrees)
SIGNS = [
    ("Cardiology", 9.0, 0.15, 13.3, 7.0),
    ("Radiology", 9.0, 0.3, 7.1, 7.0),
    ("Laboratory", 1.5, 0.3, 4.1, -10.0),
    ("Pharmacy", 6.0, 0.15, 11.3, 3.0),
    ("Pediatrics", 9.0, 0.6, -24.8, 7.0),
    ("Neurology", 1.5, 0.15, -4.2, -10.0),
    ("Dermatology", 3.0, 0.15, -23.6, -3.0),
    ("Oncology", 1.5, 0.6, 28.4, -10.0),
    ("Orthopedics", 3.0, 0.6, -30.9, -3.0),
    ("Urology", 6.0, 0.6, 22.3, 3.0),
    ("Blood donation", 3.0, 0.3, -29.8, -3.0),
    ("Cafeteria", 6.0, 0.3, 13.1, 3.0)]
SIGMA, SNAP = 0.8, 2.0   # scatter of the ray and half-width of the snap zone (deg)
print("Setup complete. Running in Colab:", IN_COLAB, "| signs:", len(SIGNS))

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 12.1": (lambda: math.sqrt(2 ** 2 + 0.5 ** 2 + 4 ** 2), 1e-6),
    "Exercise 12.2": (lambda: math.degrees(math.acos(4 / 4.5)), 1e-3),
    "Exercise 12.3": (lambda: 1.6 / 0.6, 1e-3),
    "Exercise A": (lambda: (2 * (0.5 * (1 + math.erf(2 * math.degrees(math.atan(0.3 / 12)) / 1.6 / math.sqrt(2)))) - 1) ** 2, 1e-4),
    "Exercise B": (lambda: 0.6 + 40 * (0.6 - 0.467) ** 2, 0.005),
    "Exercise C": (lambda: 24 / 1.4 + 180 / 60, 0.05),
    "Exercise D": (lambda: 2 * 3.74, 0.01),
    "Exercise E": (lambda: 2 * (1 - 0.5 * (1 + math.erf(3.0 / (2 * 1.46) / math.sqrt(2)))), 1e-3),
    "Exercise F": (lambda: math.prod(1 - 2 * (1 - 0.5 * (1 + math.erf(s / (2 * g) / math.sqrt(2)))) for s, g in zip([3.0, 2.5, 3.0, 4.0, 2.5, 3.0], [0.44, 0.62, 0.92, 1.07, 1.28, 1.46])), 1e-3),
}

## Python step 12: Vectors in three dimensions

A 3D interface places the user, the signs and the rays in one space, and every question about them is a question about vectors. This step stores points and directions as NumPy arrays of three numbers &#91;[1](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-12/lecture.html#ref-1)&#93;. It computes lengths and unit vectors, angles with the dot product, the normal of a surface with the cross product and the point where a ray meets a plane. The sections of the week use these pieces for rays, virtual hands, routes and arrows.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "SIGNS" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-12/"
               "NB12_3d_ui_vr_ar.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### Vectors, lengths and unit vectors

In [ ]:
# Python practice: a small example for learning the language
eye = np.array([0.0, 1.6, 0.0])     # the eyes of a patient (m)
sign = np.array([3.0, 2.2, -6.0])   # the centre of a hanging sign
v = sign - eye                      # from the eyes to the sign
print(v, "| length:", np.linalg.norm(v).round(3))
u = v / np.linalg.norm(v)           # a unit vector: the direction alone
print(u.round(3), "| length:", np.linalg.norm(u).round(3))

The three numbers are x to the right, y upwards and z towards the viewer, so looking ahead means looking along $-z$, as in the 3D scenes of the course. Subtracting two points gives the vector from one to the other. `np.linalg.norm` returns its length $\sqrt{x^2 + y^2 + z^2}$, here the distance to the sign in metres. Dividing by the length keeps the direction and sets the length to 1.

### The dot product and the angle between two directions

In [ ]:
# Python practice: a small example for learning the language
gaze = np.array([0.0, 0.0, -1.0])  # looking straight ahead
cos_a = np.dot(gaze, u)             # both vectors have length 1
print(round(cos_a, 3), "| angle:",
      round(np.degrees(np.arccos(cos_a)), 1), "degrees")

The dot product multiplies the coordinates pair by pair and adds them up. It equals $|a|\,|b|\cos\theta$, so for two unit vectors it is the cosine of the angle between them. `np.arccos` turns the cosine back into the angle, in radians, and `np.degrees` converts it. The sign lies 27.0 degrees away from the line of sight, so the patient must turn the head or the eyes.

### The cross product and the normal of a surface

In [ ]:
# Python practice: a small example for learning the language
right = np.array([1.0, 0.0, 0.0])    # along the floor to the right
ahead = np.array([0.0, 0.0, -1.0])   # along the floor, straight ahead
up = np.array([0.0, 1.0, 0.0])
print("normal of the floor:", np.cross(right, ahead) + 0.0)   # + 0.0: no -0.0
print("normal of a side wall:", np.cross(ahead, up) + 0.0)

The cross product of two vectors is a third vector at right angles to both. For two directions that lie in a surface, it is the *normal* of the surface: Here $(0, 1, 0)$, straight up from the floor. The second normal, $(1, 0, 0)$, points away from a wall on the left of the corridor. The order matters, because swapping the two vectors turns the normal around. Adding 0.0 turns the value -0.0 of the computation into 0.0.

### A ray and the plane it hits

In [ ]:
# Python practice: a small example for learning the language
p0 = np.array([0.0, 0.0, -6.0])     # a point of the wall with the sign
normal = np.array([0.0, 0.0, 1.0])  # the wall faces the patient
t = np.dot(p0 - eye, normal) / np.dot(u, normal)
hit = eye + t * u                   # the point where the ray meets the wall
print("t =", round(t, 3), "m | hit:", hit.round(3))

A ray starts at a point $o$ and runs along a direction $u$, so its points are $p(t) = o + t\,u$. A plane holds every point $p$ with $(p - p_0) \cdot n = 0$. Putting the ray into the plane gives $t = \frac{(p_0 - o) \cdot n}{u \cdot n}$. With a unit vector $u$, $t$ is the distance along the ray in metres. The ray of this example ends at the centre of the sign.

**Quick check.** Why must the direction be a unit vector for `t` to be a distance in metres?

<details><summary>Answer</summary>

The ray moves $t\,|u|$ metres for a given $t$. Only when $|u| = 1$ does $t$ count metres directly.

</details>

### Exercises for Python step 12

**Exercise 12.1.** A sign hangs at (2, 2.1, -4), and the eyes are at (0, 1.6, 0). Store the distance from the eyes to the sign in `dist`.

In [ ]:
dist = None   # your answer

check("Exercise 12.1", dist)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
dist = np.linalg.norm(np.array([2, 2.1, -4]) - eye)

check("Exercise 12.1", dist)

**Exercise 12.2.** Store the angle in degrees between the line of sight `gaze` and the direction from the eyes to the same sign in `angle`.

In [ ]:
angle = None   # your answer

check("Exercise 12.2", angle)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
w = np.array([2, 2.1, -4]) - eye
angle = np.degrees(np.arccos(np.dot(gaze, w / np.linalg.norm(w))))

check("Exercise 12.2", angle)

**Exercise 12.3.** A ray leaves the eyes along $(0, -0.6, -0.8)$. At which distance `t_floor` does it meet the floor, the plane through $(0, 0, 0)$ with the normal $(0, 1, 0)$?

In [ ]:
t_floor = None   # your answer

check("Exercise 12.3", t_floor)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
d = np.array([0.0, -0.6, -0.8])
fl = np.array([0.0, 1.0, 0.0])
t_floor = np.dot(np.zeros(3) - eye, fl) / np.dot(d, fl)

check("Exercise 12.3", t_floor)

## 1. Pointing with a ray &#91;[7](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-12/lecture.html#ref-7)&#93;

*Lecture: [Selecting at a distance: Pointing with a ray](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-12/lecture.html#ray-casting)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "hit" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-12/"
               "NB12_3d_ui_vr_ar.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A ray selects whatever it meets first, at any distance, but a trembling hand moves it by an angle. Python computes the angular width of the twelve signs of the atrium and the probability that the ray lands on each of them. It checks the formula with simulated rays and then analyses your clicks of Part A.

**From the formulas to the code.** `angular_width(w, d)` computes $\omega = 2 \arctan\frac{w}{2d}$ in degrees. `p_hit(om)` computes $P = \left(2\,\Phi\!\left(\frac{\omega}{2\sigma}\right) - 1\right)^2$ with $\sigma = 0.8$ degrees, and `p_hit_snap(om)` replaces $\omega$ by $\max(\omega, 2s)$ with the snap zone $s = 2$ degrees. `phi(z)` is the cumulative standard normal distribution $\Phi$.

In [ ]:
def phi(z):
    """Cumulative standard normal distribution."""
    return 0.5 * (1 + math.erf(z / math.sqrt(2)))

def angular_width(w, d):
    """Visual angle of a sign of width w (m) at distance d (m), in degrees."""
    return 2 * math.degrees(math.atan(w / (2 * d)))

def p_hit(om, sigma=SIGMA):
    """Probability that the ray lands on a square sign of om degrees."""
    p1 = 2 * phi(om / (2 * sigma)) - 1
    return p1 ** 2

def p_hit_snap(om, sigma=SIGMA, snap=SNAP):
    """The same with a snap zone of 2 * snap degrees around the centre."""
    return p_hit(max(om, 2 * snap), sigma)

rows = []
for name, d, w, az, el in SIGNS:
    om = angular_width(w, d)
    rows.append((name, d, w, round(om, 3), round(p_hit(om), 3),
                 round(p_hit_snap(om), 3)))
table = pd.DataFrame(rows, columns=["sign", "d (m)", "w (m)", "omega (deg)",
                                    "P ray", "P snap"])
print(table.to_string(index=False))
print("mean over the twelve signs: ray", round(table["P ray"].mean(), 3),
      "| snap", round(table["P snap"].mean(), 3))

**Python note.** `math.erf` is the error function of the standard library, from which the normal distribution follows. `DataFrame.to_string(index=False)` prints the whole table without the row numbers.

**Exercise A.** A sign 0.3 m wide hangs 6 m away. Store the probability that the plain ray hits it in `p_cafe`.

In [ ]:
p_cafe = None   # your answer

check("Exercise A", p_cafe)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_cafe = p_hit(angular_width(0.3, 6))

check("Exercise A", p_cafe)

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n = 100_000
for name, d, w, az, el in SIGNS[:2]:
    om = angular_width(w, d)
    zone = max(om, 2 * SNAP)
    dx, dy = rng.normal(0, SIGMA, (2, n))     # scatter of the ray (deg)
    plain = (np.abs(dx) <= om / 2) & (np.abs(dy) <= om / 2)
    snapped = (np.abs(dx) <= zone / 2) & (np.abs(dy) <= zone / 2)
    print(f"{name}: simulated {plain.mean():.3f} (model {p_hit(om):.3f}), "
          f"snapping {snapped.mean():.3f} (model {p_hit_snap(om):.3f})")

**Python note.** `rng.normal(0, SIGMA, (2, n))` draws two rows of random angles at once, one for each direction, and `dx, dy = ...` unpacks the two rows. A ray hits the square when both angles stay within half its width.

In [ ]:
# Figure: the hit probability against the distance, and 300 rays at a sign
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
dist = np.linspace(0.5, 12, 200)
for w, col in zip([0.15, 0.3, 0.6], ["#B53F2E", "#C98510", "#2E8B57"]):
    oms = [angular_width(w, d) for d in dist]
    ax[0].plot(dist, [p_hit(o) for o in oms], color=col, label=f"{w} m, ray")
    ax[0].plot(dist, [p_hit_snap(o) for o in oms], "--", color=col,
               label=f"{w} m, snapping")
ax[0].set_xlabel("distance of the sign (m)")
ax[0].set_ylabel("hit at the first click")
ax[0].legend(fontsize=7, ncol=2)
name, d, w, az, el = SIGNS[0]
om = angular_width(w, d)
dx, dy = rng.normal(0, SIGMA, (2, 300))
inside = (np.abs(dx) <= om / 2) & (np.abs(dy) <= om / 2)
ax[1].scatter(dx, dy, s=6, c=np.where(inside, "#2E8B57", "#B53F2E"))
ax[1].add_patch(plt.Rectangle((-om / 2, -om / 2), om, om, fill=False, lw=2))
ax[1].add_patch(plt.Rectangle((-SNAP, -SNAP), 2 * SNAP, 2 * SNAP, fill=False,
                              ls="--", color="#5A4BB5"))
ax[1].set_aspect("equal")
ax[1].set_title(f"{name}: {w} m at {d} m, {om:.2f} degrees", fontsize=9)
ax[1].set_xlabel("degrees to the right")
ax[1].set_ylabel("degrees up")
plt.tight_layout()
plt.show()

**Python note.** Each dot on the right is one click of the ray. Green dots land on the sign, the black square, and the dashed violet square is the snap zone of round 2.

**Paste your results.** Copy the results of Part A of the interactive lab and paste them between the triple quotes of `MY_SELECTION` below. If you leave it empty, the code uses the results of a typical student.

In [ ]:
MY_SELECTION = """
"""
DEFAULT_SELECTION = """
round,technique,label,d_m,w_m,omega_deg,amp_deg,ms,hit,err_deg
1,ray,Cardiology,9.0,0.15,0.955,15.0,1274,1,0.17
1,ray,Radiology,9.0,0.3,1.91,6.2,952,1,0.1
1,ray,Laboratory,1.5,0.3,11.421,17.3,807,1,5.44
1,ray,Pharmacy,6.0,0.15,1.432,14.8,1168,0,0.34
1,ray,Pediatrics,9.0,0.6,3.818,36.2,1280,1,0.84
1,ray,Neurology,1.5,0.15,5.725,26.6,1088,1,0.09
1,ray,Dermatology,3.0,0.15,2.864,20.5,1201,1,0.11
1,ray,Oncology,1.5,0.6,22.62,52.1,821,1,4.35
1,ray,Orthopedics,3.0,0.6,11.421,59.3,1028,1,6.4
1,ray,Urology,6.0,0.6,5.725,53.5,981,1,1.28
1,ray,Blood donation,3.0,0.3,5.725,52.4,1216,1,0.35
1,ray,Cafeteria,6.0,0.3,2.864,43.3,1165,1,1.22
2,snap,Cardiology,9.0,0.15,0.955,15.0,928,1,0.44
2,snap,Radiology,9.0,0.3,1.91,6.2,519,1,0.19
2,snap,Laboratory,1.5,0.3,11.421,17.3,620,1,2.9
2,snap,Pharmacy,6.0,0.15,1.432,14.8,954,1,0.07
2,snap,Pediatrics,9.0,0.6,3.818,36.2,1229,1,0.55
2,snap,Neurology,1.5,0.15,5.725,26.6,925,1,0.36
2,snap,Dermatology,3.0,0.15,2.864,20.5,923,1,0.42
2,snap,Oncology,1.5,0.6,22.62,52.1,699,1,0.47
2,snap,Orthopedics,3.0,0.6,11.421,59.3,942,1,0.63
2,snap,Urology,6.0,0.6,5.725,53.5,1049,1,1.4
2,snap,Blood donation,3.0,0.3,5.725,52.4,1066,1,1.46
2,snap,Cafeteria,6.0,0.3,2.864,43.3,1278,1,1.61"""
src = MY_SELECTION.strip() or DEFAULT_SELECTION.strip()
print("Data:", "your own clicks from the interactive lab" if MY_SELECTION.strip()
      else "the results of a typical student")
sel = pd.read_csv(io.StringIO(src))
sel["model"] = [p_hit(o) if t == "ray" else p_hit_snap(o)
                for o, t in zip(sel.omega_deg, sel.technique)]
for tech, part in sel.groupby("technique", sort=False):
    print(f"{tech}: {part.hit.sum()} of {len(part)} hits | the model expects "
          f"{part.model.sum():.2f} | mean time {part.ms.mean():.0f} ms")
eff = np.where(sel.technique == "ray", sel.omega_deg,
               np.maximum(sel.omega_deg, 2 * SNAP))
sel["ID"] = np.log2(sel.amp_deg / eff + 1)
b, a = np.polyfit(sel.ID, sel.ms, 1)
print(f"time = {a:.0f} + {b:.0f} x ID (ms), with ID = log2(A / omega + 1)")

**Python note.** `amp_deg` is the angle from the previous sign to the current one, the amplitude $A$ of the movement. With snapping, the effective width is the snap zone, so the index of difficulty uses `np.maximum(omega, 2 * SNAP)`. `np.polyfit(x, y, 1)` returns the slope first and then the intercept.

**Explore.** Move the sliders: The code prints the angular width of a sign and the hit probabilities of both rays. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as widgets

def explore(d=9.0, w=0.15, sigma=0.8):
    om = angular_width(w, d)
    print(f"omega {om:.3f} degrees | ray {p_hit(om, sigma):.3f} | "
          f"snapping {p_hit_snap(om, sigma):.3f}")

widgets.interact(explore,
                 d=widgets.FloatSlider(value=9.0, min=0.5, max=15, step=0.5),
                 w=widgets.FloatSlider(value=0.15, min=0.05, max=1.0, step=0.05),
                 sigma=widgets.FloatSlider(value=0.8, min=0.2, max=2.0, step=0.1));

**Observe and reflect.** Did you hit more far signs than the model expects in round 1? The scatter of the game swings like a tremor, so a click can wait for the right moment. What does that cost in time?

## 2. Reaching with a virtual hand: Go-Go &#91;[8](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-12/lecture.html#ref-8)&#93;

*Lecture: [Reaching with a virtual hand: The Go-Go technique](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-12/lecture.html#go-go)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "p_hit_snap" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-12/"
               "NB12_3d_ui_vr_ar.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** A virtual hand grabs objects as a real hand does, but it reaches only as far as the arm. The Go-Go technique stretches the virtual arm beyond a threshold. Python computes the reach of the clinic's setting and the gain that comes with it. It then compares the virtual hand with the ray on the signs that the hand reaches.

**From the formulas to the code.** `gogo(r)` computes $R_v = R_r$ for $R_r < D$ and $R_v = R_r + k(R_r - D)^2$ otherwise, with $D = \tfrac{2}{3}$ of the arm and $k = 40$ per metre. `gain(r)` is the slope $1 + 2k(R_r - D)$, and `real_for(R)` solves the quadratic for the real distance. The virtual hand wavers by `gain(r) * HAND_SD`.

In [ ]:
ARM, K, HAND_SD = 0.7, 40.0, 0.005   # m, per metre, waver of the hand (m)
D_LIN = round(2 / 3 * ARM, 3)               # the end of the linear part

def gogo(r):
    """Distance of the virtual hand for the real hand at r (m)."""
    return r if r < D_LIN else r + K * (r - D_LIN) ** 2

def gain(r):
    """How many times faster the virtual hand moves than the real one."""
    return 1.0 if r < D_LIN else 1 + 2 * K * (r - D_LIN)

def real_for(rv):
    """Real distance that places the virtual hand at rv (m)."""
    if rv < D_LIN:
        return rv
    c = D_LIN - rv                          # K u^2 + u + c = 0, u = r - D
    return D_LIN + (-1 + math.sqrt(1 - 4 * K * c)) / (2 * K)

for r in (0.3, D_LIN, 0.6, ARM):
    print(f"real {r:.3f} m -> virtual {gogo(r):.3f} m, gain {gain(r):.2f}")
print("the round trip:", round(real_for(gogo(0.6)), 3))

**Python note.** `r if r < D_LIN else ...` is a conditional expression: One value when the test holds, the other when it does not. `real_for` uses the formula for the roots of a quadratic and keeps the positive root.

**Exercise B.** Store the distance of the virtual hand for a real hand 0.6 m from the body in `rv`.

In [ ]:
rv = None   # your answer

check("Exercise B", rv)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
rv = gogo(0.6)

check("Exercise B", rv)

In [ ]:
reach = gogo(ARM)
print(f"D = {D_LIN} m | the stretched arm reaches {reach:.3f} m, "
      f"with a gain of {gain(ARM):.2f}")
rows = []
for name, d, w, az, el in SIGNS:
    if d > reach:
        continue
    r = real_for(d)
    sd = gain(r) * HAND_SD                  # waver of the virtual hand (m)
    p_hand = (2 * phi(w / (2 * sd)) - 1) ** 2
    rows.append((name, d, w, round(r, 3), round(gain(r), 2),
                 round(sd * 1000, 1), round(p_hand, 3),
                 round(p_hit(angular_width(w, d)), 3)))
print(pd.DataFrame(rows, columns=["sign", "d", "w", "real r", "gain", "waver (mm)",
                                  "P hand", "P ray"]).to_string(index=False))
far = [s[0] for s in SIGNS if s[1] > reach]
print(f"{len(far)} of 12 signs lie beyond the reach of the virtual hand")

**Python note.** `continue` skips the rest of the loop for a sign out of reach. The waver of the virtual hand is the waver of the real hand times the gain. A small movement of the real hand is stretched by the slope of the mapping.

In [ ]:
# Figure: the Go-Go mapping and the waver of the virtual hand
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
r = np.linspace(0, ARM, 200)
ax[0].plot(r, [gogo(x) for x in r], color="#5A4BB5", lw=2, label="Go-Go")
ax[0].plot(r, r, "--", color="#585E6E", label="one to one")
ax[0].axvline(D_LIN, color="#C98510", ls=":", label=f"D = {D_LIN} m")
for d in (1.5, 3.0):
    ax[0].axhline(d, color="#B8BDC7", lw=0.8)
ax[0].set_xlabel("real hand from the body (m)")
ax[0].set_ylabel("virtual hand (m)")
ax[0].legend(fontsize=8)
dist = np.linspace(0.3, reach, 100)
ax[1].plot(dist, [gain(real_for(x)) * HAND_SD * 1000 for x in dist],
           color="#5A4BB5", label="virtual hand")
ax[1].plot(dist, [x * math.tan(math.radians(SIGMA)) * 1000 for x in dist],
           color="#2E8B57", label="ray, at the sign")
ax[1].set_xlabel("distance of the object (m)")
ax[1].set_ylabel("waver (mm, one standard deviation)")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python note.** A ray that scatters by $\sigma$ degrees misses a point at distance $d$ by about $d \tan\sigma$ metres, so its error grows in a straight line with the distance. The error of the virtual hand grows with the gain, faster and faster.

**Observe and reflect.** The ray is more precise at every distance in the figure. Why would a designer still choose a virtual hand, for example to turn a model of the heart in the hand?

## 3. Travel: Steering and teleporting &#91;[9](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-12/lecture.html#ref-9)&#93;

*Lecture: [Travel in a virtual hospital: Steering and teleporting](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-12/lecture.html#travel)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "real_for" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-12/"
               "NB12_3d_ui_vr_ar.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** In a virtual tour, the body stays in the chair while the view moves. Python computes the time of the tour and its seconds of smooth visual motion for steering and for teleporting. It then uses vectors to find the direction back to the lift and the error of your pointing in Part C.

**From the formulas to the code.** `tour_steer` computes $T = \frac{L}{v} + \frac{\Theta}{\omega}$, the length of the route over the speed plus the turns over the rate of turning. All of it is smooth visual motion. `tour_teleport` counts $\lceil L_i / r \rceil$ jumps per corridor of length $L_i$ and range $r$, at 1.2 s each, plus snap turns of 45 degrees at 0.4 s each. `pointing_error` computes $\theta = \arccos(u \cdot v)$ for two unit vectors.

In [ ]:
LEGS, TURNS = [10, 6, 8], [90, 90]         # corridors (m) and turns (degrees)
SPEED, TURN_RATE = 1.4, 60.0           # m/s and degrees per second
TP_RANGE, TP_AIM = 4.0, 1.2           # m and s per teleport
SNAP_TURN, SNAP_COST = 45, 0.4          # degrees and s per snap turn

def tour_steer(legs=LEGS, turns=TURNS):
    """Steering: the view glides and turns all the time."""
    t = sum(legs) / SPEED + sum(turns) / TURN_RATE
    return dict(time=round(t, 1), motion=round(t, 1), jumps=0, snaps=0)

def tour_teleport(legs=LEGS, turns=TURNS):
    """Teleporting: jumps of up to TP_RANGE and snap turns, no gliding."""
    jumps = sum(math.ceil(leg / TP_RANGE) for leg in legs)
    snaps = sum(t // SNAP_TURN for t in turns)
    t = jumps * TP_AIM + snaps * SNAP_COST
    return dict(time=round(t, 1), motion=0.0, jumps=jumps, snaps=snaps)

print("steer:   ", tour_steer())
print("teleport:", tour_teleport())

**Python note.** `math.ceil` rounds up, because a corridor of 10 m needs three jumps of at most 4 m. `//` divides and drops the remainder, so a turn of 90 degrees takes two snap turns of 45.

**Exercise C.** The steered tour covers 24 m at 1.4 m/s and two turns of 90 degrees at 60 degrees per second. Compute its time without the helper functions and store it in `t_steer`.

In [ ]:
t_steer = None   # your answer

check("Exercise C", t_steer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
t_steer = 24 / 1.4 + 180 / 60

check("Exercise C", t_steer)

In [ ]:
headings = np.radians(np.cumsum([0] + [90, -90]))   # first corridor, right, left
pos = np.zeros(2)                       # x across, y along the first corridor (m)
for leg, hd in zip(LEGS, headings):
    pos = pos + leg * np.array([np.sin(hd), np.cos(hd)])
back = -pos / np.linalg.norm(pos)        # unit vector from room 12 to the lift
bearing = np.degrees(np.arctan2(back[0], back[1])) % 360
print("room 12 lies at", pos.round(2), "m | the lift at the bearing",
      round(bearing, 1), "degrees")

def pointing_error(pointed, true):
    """Angle between two horizontal bearings, from their dot product."""
    a, b = np.radians([pointed, true])
    u, v = np.array([np.sin(a), np.cos(a)]), np.array([np.sin(b), np.cos(b)])
    return math.degrees(math.acos(np.clip(u @ v, -1, 1)))

print("pointing at 205 degrees misses by", round(pointing_error(205, bearing), 1),
      "degrees")

**Python note.** A bearing is measured clockwise from the first corridor, so its unit vector is $(\sin a, \cos a)$. `np.arctan2(x, y)` returns the bearing of a vector in every quadrant, and `% 360` keeps it between 0 and 360. `np.clip` guards `acos` against a cosine of 1.0000001 from rounding.

**Paste your results.** Copy the results of Part C of the interactive lab and paste them between the triple quotes of `MY_TRAVEL` below. If you leave it empty, the code uses the results of a typical student.

In [ ]:
MY_TRAVEL = """
"""
DEFAULT_TRAVEL = """
round,mode,time_s,moving_s,jumps,snaps,path_m,pointed_deg,true_deg
1,steer,31.4,27.9,0,0,25.3,205.0,198.4
2,teleport,16.2,0.0,8,4,25.9,152.0,198.4"""
src = MY_TRAVEL.strip() or DEFAULT_TRAVEL.strip()
print("Data:", "your own tours from the interactive lab" if MY_TRAVEL.strip()
      else "the results of a typical student")
trv = pd.read_csv(io.StringIO(src))
trv["error_deg"] = [round(pointing_error(p, t), 1)
                    for p, t in zip(trv.pointed_deg, trv.true_deg)]
model = {"steer": tour_steer(), "teleport": tour_teleport()}
trv["model_time_s"] = [model[m]["time"] for m in trv["mode"]]
print(trv[["mode", "time_s", "model_time_s", "moving_s", "jumps", "snaps",
           "error_deg"]].to_string(index=False))

**Python note.** The model gives the fastest possible tour, with no hesitation and a straight path. Your times show how much the search for the way and the aiming add to it.

In [ ]:
# Figure: the route with the pointed directions, and the times of the tours
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
path = np.vstack([np.zeros(2), np.cumsum(
    [leg * np.array([np.sin(h), np.cos(h)]) for leg, h in zip(LEGS, headings)],
    axis=0)])
ax[0].plot(path[:, 0], path[:, 1], color="#585E6E", lw=6, alpha=0.35)
ax[0].plot(*path[0], "s", color="#2F5D8A", ms=10, label="lift")
ax[0].plot(*path[-1], "o", color="#B53F2E", ms=10, label="room 12")
for (_, row), col in zip(trv.iterrows(), ["#2E8B57", "#C98510"]):
    a = math.radians(row.pointed_deg)
    ax[0].arrow(*path[-1], 6 * math.sin(a), 6 * math.cos(a), width=0.25, color=col,
                label=f"{row['mode']}: {row.error_deg} degrees off")
ax[0].set_aspect("equal")
ax[0].legend(fontsize=7, loc="lower right")
ax[0].set_xlabel("x (m)")
ax[0].set_ylabel("y (m)")
x = np.arange(len(trv))
ax[1].bar(x - 0.2, trv.time_s, 0.4, color="#5A4BB5", label="whole tour")
ax[1].bar(x + 0.2, trv.moving_s, 0.4, color="#B53F2E", label="smooth motion")
ax[1].set_xticks(x, trv["mode"])
ax[1].set_ylabel("seconds")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python note.** `np.cumsum(..., axis=0)` adds the vectors of the corridors one after another, so each row is a corner of the route. `ax.arrow(x, y, dx, dy)` draws a pointed direction from room 12.

**Observe and reflect.** In which round did you point back to the lift more accurately? Round 2 already knew the route, so how would you change the order of the rounds to compare the techniques fairly?

## 4. Presence and cybersickness: The sickness questionnaire &#91;[13](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-12/lecture.html#ref-13)&#93;

*Lecture: [Presence and cybersickness](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-12/lecture.html#presence-and-cybersickness)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "pointing_error" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-12/"
               "NB12_3d_ui_vr_ar.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Visual motion without bodily motion can make users sick, and the Simulator Sickness Questionnaire (SSQ) measures how sick. Python scores the questionnaire of one patient after the steered tour. It then simulates the pilot of the virtual tour, 24 patients with each technique, and compares the totals.

**From the formulas to the code.** `ssq(ratings)` adds the ratings of the symptoms of each subscale to the raw sums $[N]$, $[O]$ and $[D]$. It multiplies them by 9.54, 7.58 and 13.92, and computes the total score (TS) as $([N] + [O] + [D]) \times 3.74$. A symptom can count for two subscales.

In [ ]:
SSQ_ITEMS = [   # symptom and the subscales it counts for
    ("general discomfort", "NO"), ("fatigue", "O"),
    ("headache", "O"), ("eyestrain", "O"),
    ("difficulty focusing", "OD"), ("increased salivation", "N"),
    ("sweating", "N"), ("nausea", "ND"),
    ("difficulty concentrating", "NO"), ("fullness of head", "D"),
    ("blurred vision", "OD"), ("dizzy (eyes open)", "D"),
    ("dizzy (eyes closed)", "D"), ("vertigo", "D"),
    ("stomach awareness", "N"), ("burping", "N")]
W_SSQ = {"N": 9.54, "O": 7.58, "D": 13.92}

def ssq(ratings):
    """Raw sums and scores of the SSQ from 16 ratings of 0 to 3."""
    raw = {s: sum(r for r, (_, subs) in zip(ratings, SSQ_ITEMS) if s in subs)
           for s in "NOD"}
    scores = {s: round(raw[s] * W_SSQ[s], 2) for s in "NOD"}
    scores["TS"] = round((raw["N"] + raw["O"] + raw["D"]) * 3.74, 2)
    return raw, scores

EXAMPLE = [1, 1, 0, 2, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0]
raw, scores = ssq(EXAMPLE)
for (name, subs), r in zip(SSQ_ITEMS, EXAMPLE):
    if r:
        print(f"{name}: {r} ({', '.join(subs)})")
print("raw sums:", raw, "| scores:", scores)

**Python note.** The dictionary comprehension builds one raw sum for each of the letters N, O and D. The condition `if s in subs` picks the symptoms whose string of subscales contains the letter.

**Exercise D.** After the teleported tour, a patient rates fatigue and eyestrain as slight (1) and every other symptom as none (0). Store the total score in `ts`.

In [ ]:
ratings = [0] * 16
ts = None   # your answer

check("Exercise D", ts)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
ratings = [0] * 16
ratings[1] = ratings[3] = 1
ts = ssq(ratings)[1]["TS"]

check("Exercise D", ts)

In [ ]:
PILOT_P = {"steer": {"N": 0.13, "O": 0.15, "D": 0.18},
           "teleport": {"N": 0.03, "O": 0.09, "D": 0.04}}

def pilot(tech, n=24, seed=RANDOM_STATE):
    """SSQ ratings of n simulated patients after a tour."""
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(n):
        f = rng.uniform(0.3, 1.7)            # the susceptibility of the patient
        p = [min(0.9, f * max(PILOT_P[tech][s] for s in subs))
             for _, subs in SSQ_ITEMS]
        out.append(rng.binomial(3, p).tolist())
    return out

TS = {tech: np.array([ssq(r)[1]["TS"] for r in pilot(tech, seed=k)])
      for k, tech in enumerate(["steer", "teleport"])}
for tech, ts_all in TS.items():
    print(f"{tech}: mean TS {ts_all.mean():.2f}, sd {ts_all.std(ddof=1):.2f}, "
          f"above 20: {(ts_all > 20).sum()} of {len(ts_all)}")
t, p = stats.ttest_ind(TS["steer"], TS["teleport"], equal_var=False)
print(f"Welch's test: t = {t:.2f}, p = {p:.2g}")

**Python note.** `rng.binomial(3, p)` draws one rating from 0 to 3 for each symptom, with its own chance `p`. A patient with a larger susceptibility `f` has larger chances everywhere. `stats.ttest_ind(..., equal_var=False)` is Welch's test of Week 9.

In [ ]:
# Figure: the profile of one patient, and the pilot of the tour
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
keys = ["N", "O", "D", "TS"]
ax[0].bar(keys, [scores[k] for k in keys],
          color=["#2E8B57", "#2F5D8A", "#5A4BB5", "#585E6E"])
for k, v in zip(keys, [scores[k] for k in keys]):
    ax[0].text(k, v + 1, f"{v:.2f}", ha="center", fontsize=8)
ax[0].set_ylabel("score")
ax[0].set_title("one patient after the steered tour", fontsize=9)
for k, (tech, col) in enumerate([("steer", "#B53F2E"), ("teleport", "#2E8B57")]):
    jit = np.random.default_rng(k).uniform(-0.12, 0.12, len(TS[tech]))
    ax[1].scatter(k + jit, TS[tech], s=14, color=col)
    ax[1].plot([k - 0.25, k + 0.25], [TS[tech].mean()] * 2, color="#1E2433")
ax[1].set_xticks([0, 1], ["steering", "teleporting"])
ax[1].set_ylabel("total score TS")
ax[1].set_title("the pilot: 24 patients with each technique", fontsize=9)
plt.tight_layout()
plt.show()

**Python note.** Each dot on the right is one simulated patient, and the black line marks the mean. The small random shift sideways keeps equal scores from hiding each other.

**Observe and reflect.** The scores of the pilot spread widely within each technique. Which patients of a clinic might be more susceptible, and should the virtual tour be offered to them at all?

## 5. Augmented reality: Drift and wrong turns &#91;[4](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-12/lecture.html#ref-4)&#93;

*Lecture: [Augmented reality in the corridor: Registration and wayfinding](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-12/lecture.html#augmented-reality)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "pilot" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-12/"
               "NB12_3d_ui_vr_ar.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** An arrow of augmented reality (AR) helps only where it is registered with the real corridor. Python models the drift of the arrow with the metres walked and the chance that it lands nearer a door than the turn. It compares a route without anchors with a route that resets the arrow at every turn, and then analyses your choices in Part D.

**From the formulas to the code.** `sigma_at(m)` computes the spread $\sigma = \sigma_0 + r\,m$ after $m$ metres since the last anchor, with $\sigma_0 = 0.2$ m and $r = 0.03$. `p_wrong(s, sigma)` computes $P = 2\left(1 - \Phi\!\left(\frac{s}{2\sigma}\right)\right)$ for doors $s$ metres before and after the turn. The expected wrong turns of a route add up $P$ over its turns.

In [ ]:
ROUTE = [   # metres walked to the turn, side, distance to the next door (m)
    (8, "R", 3.0),
    (6, "L", 2.5),
    (10, "R", 3.0),
    (5, "R", 4.0),
    (7, "L", 2.5),
    (6, "R", 3.0)]
SIGMA0, DRIFT = 0.2, 0.03                  # m, and m per metre walked

def sigma_at(m):
    """Spread of the arrow along the corridor m metres after an anchor."""
    return round(SIGMA0 + DRIFT * m, 2)

def p_wrong(s, sigma):
    """Chance that the arrow lies nearer a door than the turn."""
    return 2 * (1 - phi(s / (2 * sigma)))

WRONG, CLEAN = {}, {}
for mode in ("no anchors", "anchors"):
    walked, ps = 0, []
    for d, side, s in ROUTE:
        walked = d if mode == "anchors" else walked + d
        ps.append(p_wrong(s, sigma_at(walked)))
        print(f"{mode}: turn {len(ps)} after {walked} m, sigma "
              f"{sigma_at(walked):.2f} m, P(wrong) {ps[-1]:.4f}")
    WRONG[mode] = round(sum(ps), 3)
    CLEAN[mode] = round(math.prod(1 - p for p in ps), 3)
print("expected wrong turns per trip:", WRONG)
print("trips without a wrong turn:", CLEAN)

**Python note.** `walked = d if ... else walked + d` restarts the count at every anchor and keeps adding without anchors. `math.prod` multiplies the chances of a right choice at all six turns.

**Exercise E.** Without anchors, the patient has walked 42 m at the sixth turn, and the doors lie 3.0 m before and after it. Store the chance that the arrow lies nearer a door in `p6`.

In [ ]:
p6 = None   # your answer

check("Exercise E", p6)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p6 = p_wrong(3.0, sigma_at(42))

check("Exercise E", p6)

In [ ]:
rng = random.Random(1280)         # the same arrows as in Part D
Z = [round(rng.gauss(0, 1), 3) for _ in ROUTE]

def arrow_at(err, s):
    """The opening nearest to the arrow: the turn or a door."""
    if abs(err) <= s / 2:
        return "turn"
    return "door after" if err > 0 else "door before"

ARROWS = {}
for mode in ("no anchors", "anchors"):
    walked, errs = 0, []
    for (d, side, s), z in zip(ROUTE, Z):
        walked = d if mode == "anchors" else walked + d
        errs.append(round(z * sigma_at(walked), 2))
    ARROWS[mode] = errs
    print(f"{mode}: errors {errs} m")
    print("   arrows at:", [arrow_at(e, r[2]) for e, r in zip(errs, ROUTE)])

**Python note.** `random.Random(seed)` draws the same standard normal numbers `Z` as the interactive lab. Each error is such a number times the spread at that turn, so the anchors shrink every error by the same factor.

**Paste your results.** Copy the results of Part D of the interactive lab and paste them between the triple quotes of `MY_ROUTE` below. If you leave it empty, the code uses the results of a typical student.

In [ ]:
MY_ROUTE = """
"""
DEFAULT_ROUTE = """
round,mode,turn,side,spacing_m,err_m,arrow_at,choice,correct,ms
1,no anchors,1,R,3.0,0.32,turn,turn,1,2600
1,no anchors,2,L,2.5,-0.38,turn,turn,1,2980
1,no anchors,3,R,3.0,-0.12,turn,turn,1,3360
1,no anchors,4,R,4.0,0.23,turn,turn,1,2840
1,no anchors,5,L,2.5,-1.52,door before,door before,0,4720
1,no anchors,6,R,3.0,2.77,door after,turn,1,4200
2,anchors,1,R,3.0,0.32,turn,turn,1,2600
2,anchors,2,L,2.5,-0.23,turn,turn,1,2980
2,anchors,3,R,3.0,-0.06,turn,turn,1,3360
2,anchors,4,R,4.0,0.07,turn,turn,1,2840
2,anchors,5,L,2.5,-0.49,turn,turn,1,3220
2,anchors,6,R,3.0,0.72,turn,turn,1,2700"""
src = MY_ROUTE.strip() or DEFAULT_ROUTE.strip()
print("Data:", "your own route from the interactive lab" if MY_ROUTE.strip()
      else "the results of a typical student")
rte = pd.read_csv(io.StringIO(src))
for mode, part in rte.groupby("mode", sort=False):
    misled = part[part.arrow_at != "turn"]
    followed = (misled.choice == misled.arrow_at).sum()
    print(f"{mode}: wrong turns {len(part) - part.correct.sum()} of {len(part)} | "
          f"misplaced arrows {len(misled)}, followed {followed} | "
          f"mean decision {part.ms.mean() / 1000:.1f} s")

**Python note.** A misplaced arrow is one that lies nearer a door than the turn. Following it means choosing the opening where the arrow lies instead of the turn under the sign.

In [ ]:
# Figure: the spread of the arrow, and the chance of a wrong turn at each turn
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
m = np.linspace(0, 45, 100)
ax[0].plot(m, SIGMA0 + DRIFT * m, color="#5A4BB5")
for s in sorted({r[2] for r in ROUTE}):
    ax[0].axhline(s / 2, color="#B8BDC7", ls="--", lw=0.8)
    ax[0].text(45, s / 2, f" s/2 = {s / 2}", fontsize=7, va="center")
ax[0].set_xlabel("metres walked since the last anchor")
ax[0].set_ylabel("spread of the arrow (m)")
x = np.arange(1, len(ROUTE) + 1)
for k, (mode, col) in enumerate([("no anchors", "#B53F2E"),
                                 ("anchors", "#2E8B57")]):
    walked, ps = 0, []
    for d, side, s in ROUTE:
        walked = d if mode == "anchors" else walked + d
        ps.append(p_wrong(s, sigma_at(walked)))
    ax[1].bar(x + (k - 0.5) * 0.4, ps, 0.4, color=col, label=mode)
ax[1].set_xticks(x)
ax[1].set_xlabel("turn of the route")
ax[1].set_ylabel("chance of a wrong turn")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**Python note.** The dashed lines are half the distance to the next door, the spread at which wrong turns become common. Without anchors, the spread keeps growing along the whole route.

**Explore.** Move the sliders: The code prints the expected wrong turns per trip without anchors and with an anchor at every turn. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as widgets

def explore(drift=0.03, sigma0=0.2):
    for mode in ("no anchors", "anchors"):
        walked, total = 0, 0.0
        for d, side, s in ROUTE:
            walked = d if mode == "anchors" else walked + d
            total += p_wrong(s, sigma0 + drift * walked)
        print(f"{mode}: {total:.3f} wrong turns per trip")

widgets.interact(explore,
                 drift=widgets.FloatSlider(value=0.03, min=0.005, max=0.08,
                                           step=0.005),
                 sigma0=widgets.FloatSlider(value=0.2, min=0.05, max=1.0,
                                            step=0.05));

**Observe and reflect.** Did you follow a misplaced arrow in round 1, or did the sign above the turn win? What does that say about the attention that patients pay to an arrow in their glasses?

## 6. The clinic app, stage 12

*Lecture: [The clinic app, stage 12: Wayfinding in the hospital with augmented reality](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-12/lecture.html#the-clinic-app-stage-12)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "ARROWS" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-12/"
               "NB12_3d_ui_vr_ar.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 12 adds three things to the clinic app. A virtual tour comes before the visit, signs in it are selected at a distance, and arrows of augmented reality guide on the day. Python collects the results of the sections in one dictionary.

In [ ]:
STAGE12 = {}
rays = [p_hit(angular_width(w, d)) for _, d, w, _, _ in SIGNS]
snaps = [p_hit_snap(angular_width(w, d)) for _, d, w, _, _ in SIGNS]
STAGE12["signs hit at the first click, ray"] = round(float(np.mean(rays)), 3)
STAGE12["signs hit at the first click, snapping"] = round(float(np.mean(snaps)), 3)
STAGE12["the far small sign, ray"] = round(min(rays), 3)
STAGE12["the far small sign, snapping"] = round(min(snaps), 3)
STAGE12["reach of the virtual hand (m)"] = round(gogo(ARM), 3)
for tech, f in (("steering", tour_steer), ("teleporting", tour_teleport)):
    STAGE12[f"tour, {tech} (s)"] = f()["time"]
    STAGE12[f"smooth motion, {tech} (s)"] = f()["motion"]
STAGE12["mean SSQ total, steering"] = round(float(TS["steer"].mean()), 2)
STAGE12["mean SSQ total, teleporting"] = round(float(TS["teleport"].mean()), 2)
for mode in ("no anchors", "anchors"):
    STAGE12[f"expected wrong turns, {mode}"] = WRONG[mode]
    STAGE12[f"trips without a wrong turn, {mode}"] = CLEAN[mode]
for key, value in STAGE12.items():
    print(f"{key}: {value}")

**Python note.** `float(np.mean(...))` turns the NumPy number into a plain Python number, so that the dictionary prints cleanly. The values come from the functions of Sections 1 to 5.

**Exercise F.** Without anchors, the spreads at the six turns are 0.44, 0.62, 0.92, 1.07, 1.28 and 1.46 m. The doors lie 3.0, 2.5, 3.0, 4.0, 2.5 and 3.0 m from the turns. Store the chance of a trip without a wrong turn in `p_clean`.

In [ ]:
p_clean = None   # your answer

check("Exercise F", p_clean)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
p_clean = math.prod(1 - p_wrong(s, g) for s, g in zip(
    [3.0, 2.5, 3.0, 4.0, 2.5, 3.0], [0.44, 0.62, 0.92, 1.07, 1.28, 1.46]))

check("Exercise F", p_clean)

In [ ]:
# Figure: selection, the virtual tour and the arrows of stage 12
fig, ax = plt.subplots(1, 3, figsize=(11, 3.3))
ax[0].bar(["ray", "snapping"], [STAGE12["signs hit at the first click, ray"],
                                STAGE12["signs hit at the first click, snapping"]],
          color=["#B53F2E", "#2E8B57"])
ax[0].set_ylim(0, 1)
ax[0].set_ylabel("signs hit at the first click")
t = [STAGE12["tour, steering (s)"], STAGE12["tour, teleporting (s)"]]
mo = [STAGE12["smooth motion, steering (s)"],
      STAGE12["smooth motion, teleporting (s)"]]
x = np.arange(2)
ax[1].bar(x - 0.2, t, 0.4, color="#5A4BB5", label="tour")
ax[1].bar(x + 0.2, mo, 0.4, color="#B53F2E", label="smooth motion")
ax[1].set_xticks(x, ["steering", "teleporting"])
ax[1].set_ylabel("seconds")
ax[1].legend(fontsize=8)
ax[2].bar(["no anchors", "anchors"], [WRONG["no anchors"], WRONG["anchors"]],
          color=["#B53F2E", "#2E8B57"])
ax[2].set_ylabel("expected wrong turns per trip")
plt.tight_layout()
plt.show()

**Python note.** The three panels follow the three parts of stage 12: Selecting the signs, touring the hospital before the visit and following the arrows on the day.

## 7. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-12/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE12" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "human-computer-interaction-NB-lecture/main/weeks/week-12/"
               "NB12_3d_ui_vr_ar.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-7":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The models of this week apply to any application in three dimensions. Each application below has a target to select, a guided route between anchors and a virtual tour with its turns.

The values are illustrative. Set `APP` to one of the five names and run the cell.

In [ ]:
APP = "museum_guide"
# clinic, museum_guide, warehouse_picking, factory_training, campus_tour
APPS = {   # target width and distance (m), route without anchors and door
           # spacing (m), virtual tour (m) and its turns of 90 degrees
    "clinic": (0.15, 9.0, 42, 3.0, 24, 2),
    "museum_guide": (0.4, 5.0, 60, 4.0, 80, 6),
    "warehouse_picking": (0.1, 4.0, 120, 1.2, 0, 0),
    "factory_training": (0.05, 1.2, 0, 0, 30, 3),
    "campus_tour": (2.0, 40.0, 300, 8.0, 200, 8)}

w, d, route, spacing, tour, turns = APPS[APP]
om = angular_width(w, d)
print(f"{APP}: a target of {w} m at {d} m spans {om:.2f} degrees | ray "
      f"{p_hit(om):.3f}, snapping {p_hit_snap(om):.3f}")
if route:
    sg = sigma_at(route)
    print(f"  after {route} m without an anchor: spread {sg} m, a wrong turn "
          f"with probability {p_wrong(spacing, sg):.3f}")
if tour:
    st, tp = tour_steer([tour], [90] * turns), tour_teleport([tour], [90] * turns)
    print(f"  tour of {tour} m: steering {st['time']} s of smooth motion, "
          f"teleporting {tp['time']} s with {tp['jumps']} jumps")

**Python note.** A zero in the tuple means that the application has no such part, and the `if` tests skip it. The functions of Sections 1, 3 and 5 are reused unchanged.

**Observe and reflect.** Which part of your application would a user notice first: Hard selection, sickness on the tour or a wrong turn? Which one would you fix first?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Write a function that picks the sign nearest to a ray within the snap zone, and test it on two signs whose zones overlap. |
| Web development | A web page in augmented reality shows labels 0.2 m wide. Find with `angular_width` and `p_hit` the largest distance at which a plain ray still hits them 95 percent of the time. |
| Mobile development | A phone in augmented reality drifts by 2 percent of the distance walked. Find with `p_wrong` the spacing of anchors that keeps the wrong turns of the route below 0.05 per trip. |
| Game development | Tune `K` of the Go-Go technique so that the stretched arm reaches 5 m, and compute the waver of the virtual hand at that distance. |
| Embedded systems and the Internet of Things | Beacons on the ceiling reset the position of a guide every 15 m. Compute the expected wrong turns of the route of this week with `sigma_at` and `p_wrong`. |
| Data science | Simulate 1,000 pilots of the virtual tour with `pilot` and estimate how often Welch's test finds the difference between the techniques at the level of 0.05. |
| Cybersecurity | An attacker shifts the arrows of an AR guide by 2 m. Compute how much the wrong turns of the route grow, and propose a check that the app can make against the signs. |
| Health informatics | Patients who score a total of 20 or more on the SSQ should not continue a virtual tour. Compute the share of such patients for both techniques of the pilot. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-12/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/weeks/week-12).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>